# Création de Gastronobot - Agentic + RAG + LLM

Suite à l'embedding du livre présent dans le pdf "Intern_recipe.pdf", nous allons crée un ChatBot permettant de répondre à des questions relatives à ce livre précisement et pouvant en plus de cela utiliser du "ToolCalling" afin de le rendre plus complet. Voici la dernière version -

Packages & data

In [51]:
import pandas as pd
import random
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df_intern = pd.read_csv("intern_recipe_df.csv")

#Convertir la colonne embedding to np.array
df_intern["embedding"] = df_intern["embedding"].apply(lambda x: np.fromstring(x.strip("[]"), sep=' '))

#Convertion de notre embed en torch tensor
embeddings = torch.tensor(np.stack(df_intern["embedding"].tolist(), axis = 0), dtype=torch.float32).to(device)

df_intern.head()



,page_number,text,token_count,embedding
0,1,Favourite recipes from all over the world – e...,29.25,"[-0.0398610197, -0.0126488572, 0.0295582451, 0..."
1,2,PREFACE My favourite recipe The University of ...,295.00,"[0.0168058854, -0.0272758063, 0.0369669311, -0..."
2,3,PREFACE The number of international students a...,267.75,"[-0.00770807499, 0.0247783735, 0.034239579, 0...."
3,4,CORPORATE HEALTH MANAGEMENT Georg-August-Uni...,228.50,"[-0.0316110253, 0.0197241586, 0.0193728488, -0..."
4,5,INTERNATIONAL OFFICE Georg-August-Universität...,258.50,"[-0.0700615942, -0.0556043088, 0.0901097581, -..."


## Partie BM25

Permet une recherche relative à des mots clés 

In [52]:
#Creation de la partie BM25

from rank_bm25 import BM25Okapi 
# On prépare le texte pour BM25 (on coupe les phrases en listes de mots)
# On utilise ta colonne "text"
corpus = df_intern["text"].tolist()
tokenized_corpus = [doc.lower().split() for doc in corpus]

# On crée le moteur BM25
bm25 = BM25Okapi(tokenized_corpus)

query = "Iran"
tokenized_query = query.lower().split()

# BM25 donne un score à chaque ligne de ton tableau
# Plus le mot "Iran" est présent, plus le score est haut
bm25_scores = bm25.get_scores(tokenized_query)

print(bm25_scores) # Tu verras une liste de nombres (un par recette)

[0.         0.         0.         0.         0.         5.16652653
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         4.56865907
 4.6629008  0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.        ]


## Passons à la partie Retrieval

Objectif retrouver les passages clés relatifs à une requête.

In [53]:
print(embeddings.shape)
print(bm25_scores.shape)


torch.Size([79, 384])
(79,)


! Important faire attention que nos tensors/vectors crées sont au meme taille et au meme type !

On a la meme taille que précédemment c'est ok !!!

In [54]:
#On va chercher à créer un embedding pour notre requete ou l'on va appeler le meme modele que celui utilisé pour le RAG
from sentence_transformers import util, SentenceTransformer

embeddings_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4765.73it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Crééons maintenant une petit algo de recherche sémantique, On va tenter de retrouver la recette du foie gras. 

In [55]:
# --- 1. DÉFINITION DE LA REQUÊTE ---
req = "Information about Douglas Siqueira Freitas"
print(f"Requête : {req}\n")

# --- 2. CALCUL DU SCORE EMBEDDING (SÉMANTIQUE) ---
# On utilise ton code habituel, mais on convertit en numpy pour la fusion
req_embed = embeddings_model.encode(req, convert_to_tensor=True).to(device)
# On aplatit le score pour avoir une liste simple de scores
sim_scores_tensor = util.cos_sim(req_embed, embeddings).flatten()
vector_scores = sim_scores_tensor.cpu().numpy()

# --- 3. CALCUL DU SCORE BM25 (MOTS-CLÉS) ---
tokenized_query = req.lower().split()
bm25_scores = bm25.get_scores(tokenized_query)

# --- 4. NORMALISATION ET FUSION (MOYENNE) ---
# On ramène les deux scores entre 0 et 1 pour qu'ils aient le même poids
def normalize(array):
    diff = np.max(array) - np.min(array)
    if diff == 0: return array
    return (array - np.min(array)) / diff

v_norm = normalize(vector_scores)
b_norm = normalize(bm25_scores)

# On fait la moyenne des deux "cerveaux"
final_scores = (v_norm + b_norm) / 2

# --- 5. RÉCUPÉRATION DU TOP 5 HYBRIDE ---
# np.argsort donne les indices du plus petit au plus grand, on inverse avec [::-1]
topk_indices = np.argsort(final_scores)[::-1][:5]

# --- 6. AFFICHAGE DÉTAILLÉ ---
print(f"Résultats pour : '{req}'")
print("-" * 50)

for rank, idx in enumerate(topk_indices, 1):
    row = df_intern.iloc[idx]
    
    # Récupération des scores pour cette ligne précise
    sc_cos_brut = vector_scores[idx]      # Ton ancienne similarité cosinus
    sc_bm25_brut = bm25_scores[idx]      # Le score BM25 brut (ex: 12.5)
    sc_sens_norm = v_norm[idx]           # La similarité normalisée (0 à 1)
    sc_mots_norm = b_norm[idx]           # Le BM25 normalisé (0 à 1)
    sc_hybride = final_scores[idx]       # La moyenne des deux normalisés
    
    print(f"RANG {rank}")
    print(f"Page : {row['page_number']}")
    print(f"Texte : {row['text'][:150]}...") 
    
    # Affichage de tous les scores demandés
    print(f"-> Score Similarity Cos (Brut) : {sc_cos_brut:.4f}")
    print(f"-> Score BM25 (Brut)           : {sc_bm25_brut:.2f}")
    print(f"-> Score Sens (Sim Cos Norm)   : {sc_sens_norm:.4f}")
    print(f"-> Score Mots (BM25 Norm)      : {sc_mots_norm:.4f}")
    print(f"** SCORE HYBRIDE (MOYENNE) ** : {sc_hybride:.4f}")
    print("-" * 50)



Requête : Information about Douglas Siqueira Freitas

Résultats pour : 'Information about Douglas Siqueira Freitas'
--------------------------------------------------
RANG 1
Page : 18-19
Texte : INGREDIENTS PÃO DE  QUEIJO (Brazilian Cheese Bread) My favourite recipe – Brazil which can be bought in Germany 1 cup whole milk – Milch ½ cup vegetab...
-> Score Similarity Cos (Brut) : 0.2230
-> Score BM25 (Brut)           : 11.79
-> Score Sens (Sim Cos Norm)   : 0.8242
-> Score Mots (BM25 Norm)      : 1.0000
** SCORE HYBRIDE (MOYENNE) ** : 0.9121
--------------------------------------------------
RANG 2
Page : 106-107
Texte : INGREDIENTS My favourite recipe – Philippines which can be bought in Germany 5 pieces of chicken – Hähnchenstücke 1 cup soy sauce – Sojasoße ¾ cup vin...
-> Score Similarity Cos (Brut) : 0.2409
-> Score BM25 (Brut)           : 1.45
-> Score Sens (Sim Cos Norm)   : 0.8960
-> Score Mots (BM25 Norm)      : 0.1233
** SCORE HYBRIDE (MOYENNE) ** : 0.5096
---------------------

A Noter : Il existe aussi des modèles de reranking que l'on pourrait rajouter par la suite pour pouvoir encore améliorer les sorties !!!

### Crééons une fonction pour notre outil de recherche sémantique et de mots clés

In [56]:
# on va transformer le code précedent en fonction pour pouvoir l'appeler facilement avec n'importe quelle requete

def hybrid_search(req, top_k=2):
    # --- 1. CALCUL DU SCORE EMBEDDING (SÉMANTIQUE) ---
    req_embed = embeddings_model.encode(req, convert_to_tensor=True).to(device)
    sim_scores_tensor = util.cos_sim(req_embed, embeddings).flatten()
    vector_scores = sim_scores_tensor.cpu().numpy()

    # --- 2. CALCUL DU SCORE BM25 (MOTS-CLÉS) ---
    tokenized_query = req.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)

    # --- 3. NORMALISATION ET FUSION (MOYENNE) ---
    def normalize(array):
        diff = np.max(array) - np.min(array)
        if diff == 0: return array
        return (array - np.min(array)) / diff

    v_norm = normalize(vector_scores)
    b_norm = normalize(bm25_scores)
    final_scores = (v_norm + b_norm) / 2

    # --- 4. RÉCUPÉRATION DU TOP K HYBRIDE ---
    topk_indices = np.argsort(final_scores)[::-1][:top_k]
    
    # --- 5. AFFICHAGE DÉTAILLÉ ---
    print(f"Résultats pour : '{req}'")
    print("-" * 50)

    for rank, idx in enumerate(topk_indices, 1):
        row = df_intern.iloc[idx]
        
        sc_cos_brut = vector_scores[idx]      
        sc_bm25_brut = bm25_scores[idx]      
        sc_sens_norm = v_norm[idx]           
        sc_mots_norm = b_norm[idx]           
        sc_hybride = final_scores[idx]       
        
        print(f"RANG {rank}")
        print(f"Page : {row['page_number']}")
        print(f"Texte : {row['text']}") 
        print(f"-> Score Similarity Cos (Brut) : {sc_cos_brut:.4f}")
        print(f"-> Score BM25 (Brut)           : {sc_bm25_brut:.2f}")
        print(f"-> Score Sens (Sim Cos Norm)   : {sc_sens_norm:.4f}")
        print(f"-> Score Mots (BM25 Norm)      : {sc_mots_norm:.4f}")
        print(f"** SCORE HYBRIDE (MOYENNE) ** : {sc_hybride:.4f}")
        print("-" * 50)

# Exemple d'utilisation de la fonction
hybrid_search("Information about Douglas Siqueira Freitas", top_k=5)


Résultats pour : 'Information about Douglas Siqueira Freitas'
--------------------------------------------------
RANG 1
Page : 18-19
Texte : INGREDIENTS PÃO DE  QUEIJO (Brazilian Cheese Bread) My favourite recipe – Brazil which can be bought in Germany 1 cup whole milk – Milch ½ cup vegetable oil – Pflanzenöl 1 tsp salt – Salz 2 cups tapioca or cassava flour – Tapioka-Stärke 2 large eggs – Eier 1 to 1½ cups grated Parmesan cheese  – Parmesan Käse Equipment Medium saucepan Wooden spoon Standing mixer with paddle attachment  (or mixing bowl and elbow grease) 2 baking sheets Parchment paper or silicone baking mats Douglas Siqueira Freitas Brazilian Doctorate student  Department of crop science, plant nutrition  and phylogenetic physiology April – October 2017  (possible extension for more 6 months) Research project:  Nickel in soybean genotypes NOTES to the reciepe Tapioca flour: Sour cassava flour or sour  tapioca flour can be tricky to find. Look for it  at Latin American markets. Plain

Changement code pour avoir une liste de text 

In [57]:
# on va transformer le code précedent en fonction pour pouvoir l'appeler facilement avec n'importe quelle requete

def hybrid_search(req, top_k=3):
    # --- 1. CALCUL DU SCORE EMBEDDING (SÉMANTIQUE) ---
    req_embed = embeddings_model.encode(req, convert_to_tensor=True).to(device)
    sim_scores_tensor = util.cos_sim(req_embed, embeddings).flatten()
    vector_scores = sim_scores_tensor.cpu().numpy()

    # --- 2. CALCUL DU SCORE BM25 (MOTS-CLÉS) ---
    tokenized_query = req.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)

    # --- 3. NORMALISATION ET FUSION (MOYENNE) ---
    def normalize(array):
        diff = np.max(array) - np.min(array)
        if diff == 0: return array
        return (array - np.min(array)) / diff

    v_norm = normalize(vector_scores)
    b_norm = normalize(bm25_scores)
    final_scores = (v_norm + b_norm) / 2

    # --- 4. RÉCUPÉRATION DU TOP K HYBRIDE ---
    topk_indices = np.argsort(final_scores)[::-1][:top_k]
    
    # --- 5. AFFICHAGE DÉTAILLÉ ---


    #6 création d'un diconnaire pour stocker les résultats
    list_res = []

    for rank, idx in enumerate(topk_indices, 1):
        row = df_intern.iloc[idx]
        
        sc_cos_brut = vector_scores[idx]      
        sc_bm25_brut = bm25_scores[idx]      
        sc_sens_norm = v_norm[idx]           
        sc_mots_norm = b_norm[idx]           
        sc_hybride = final_scores[idx]       


        dico = {
            "page" : row['page_number'],
            "text" : row['text'],
             }
        
        list_res.append(dico)

    return list_res
# Exemple d'utilisation de la fonction
hybrid_search("Information about Douglas Siqueira Freitas", top_k=5)


[{'page': '18-19',
  'text': 'INGREDIENTS PÃO DE  QUEIJO (Brazilian Cheese Bread) My favourite recipe – Brazil which can be bought in Germany 1 cup whole milk – Milch ½ cup vegetable oil – Pflanzenöl 1 tsp salt – Salz 2 cups tapioca or cassava flour – Tapioka-Stärke 2 large eggs – Eier 1 to 1½ cups grated Parmesan cheese  – Parmesan Käse Equipment Medium saucepan Wooden spoon Standing mixer with paddle attachment  (or mixing bowl and elbow grease) 2 baking sheets Parchment paper or silicone baking mats Douglas Siqueira Freitas Brazilian Doctorate student  Department of crop science, plant nutrition  and phylogenetic physiology April – October 2017  (possible extension for more 6 months) Research project:  Nickel in soybean genotypes NOTES to the reciepe Tapioca flour: Sour cassava flour or sour  tapioca flour can be tricky to find. Look for it  at Latin American markets. Plain tapioca flour  lacks the slightly sour, fermented flavor, but  makes a fine substitute. Storage: Leftover puff

## Application dans un LLM via API Groq

L'objectif dans un premier temps est de le faire tourner en local avec l'application Ollama et Llama 3.2

Ce qu'on va chercher à faire mtn c'est de créer un code qui va prendre notre question l'augmenter avec la méthode de rétrieval et ensuite envoye Question + Context à Ollamma

In [58]:
#Groq utilise la version de open AI pour prendre en compte les messages donc des modifications du script vis à vis de Ollama sont nécessaires
def augmentation_groq(req, top) :
    message = []
    
    #1. ON va chercher à avoir le top 5 des éléments qui pourront aider à répondre à la question
    top_5 = hybrid_search(req, top_k= top)

    
    #2. On va créer le dictionnaire relatif à comment le LLM doit répondre à la question en se basant sur les éléments trouvés
    context = "You are an assistant specializing in culinary recipes from around the world, created by university researchers.\n"
    context += "Based on these excerpts, please provide a detailed answer to the question and use only the information provided and give the page numbers.\n\n"
    context += "Do not try to give more information than what is provided.\n\n"
    context += "If you cannot find an answer to the question, do not try to invent one.\n\n"
    
    systeme = {
        "role": "system",
        "content": context
    }
    
    message.append(systeme)
    
    #3. On va créer le disctionnaire associés à la parti user ou l'on trouve la question ainsi que les éléments qui vont permettre de répondre à la question
    
    retr = "This is the question :" + req + "\n\n"
    retr += "Here are some excerpts to help you answer the question : \n\n"
    
    for i, e in enumerate(top_5, 1):
        retr += f"Extract {i} \n\n"
        retr += f"Pages : {e['page']} \n\n"
        retr += f"Texte : {e['text']} \n\n"
        retr += "\n\n"
    
    usere = {
        "role": "user",
        "content": retr
    }
    
    message.append(usere)
    
    return message

test = augmentation_groq("Information about Douglas Siqueira Freitas", 2)
print(test)

[{'role': 'system', 'content': 'You are an assistant specializing in culinary recipes from around the world, created by university researchers.\nBased on these excerpts, please provide a detailed answer to the question and use only the information provided and give the page numbers.\n\nDo not try to give more information than what is provided.\n\nIf you cannot find an answer to the question, do not try to invent one.\n\n'}, {'role': 'user', 'content': 'This is the question :Information about Douglas Siqueira Freitas\n\nHere are some excerpts to help you answer the question : \n\nExtract 1 \n\nPages : 18-19 \n\nTexte : INGREDIENTS PÃO DE  QUEIJO (Brazilian Cheese Bread) My favourite recipe – Brazil which can be bought in Germany 1 cup whole milk – Milch ½ cup vegetable oil – Pflanzenöl 1 tsp salt – Salz 2 cups tapioca or cassava flour – Tapioka-Stärke 2 large eggs – Eier 1 to 1½ cups grated Parmesan cheese  – Parmesan Käse Equipment Medium saucepan Wooden spoon Standing mixer with paddl

Création de la partie génération

In [ ]:
#Récupérer les clés d'API pour Groq 

API_KEYS = " Récupérer une clé API directement sur le site internent de Groq"

In [60]:
from groq import Groq


req = "give me information about Douglas Siqueira Freitas"

client = Groq(api_key= API_KEYS)

def generation_gorq(req, top):
    
    
    #1. appeler de la fonction augmentation_groq pour avoir les éléments à fournir au LLM
    messages = augmentation_groq(req, top)
    
    #2 Appeler le LLM gérer par Groq pour générer une réponse à la question en se basant sur les éléments trouvés
    completion = client.chat.completions.create(
    model="qwen/qwen3-32b",
    messages = messages
    ,
    temperature=0.4,
    max_completion_tokens=1000,
    top_p=1,
    reasoning_effort="none",
    stream=True,
    stop=None )
    
    #3. Afficher la réponse générée par le LLM
    reponse = ""
    for chunk in completion:
        reponse += chunk.choices[0].delta.content or ""
    
    return reponse
    
print(generation_gorq(req, 2))


Based on the provided information:

Douglas Siqueira Freitas is a Brazilian individual who was a doctoral student at a university in Germany, specifically in the Department of crop science, plant nutrition and phylogenetic physiology. His research project, which took place from April to October 2017 (with a potential extension for an additional 6 months), focused on "Nickel in soybean genotypes." The information provided does not specify his current status or position beyond the timeframe of his research project.


# Ajout d'Agent permettant l'amélioration du LLM

### Agent de traduction Francais -> Anglais -> Francais

In [61]:
# Dans cette nouvelle version de Gastronobot on va chercher à prendre la question en entrée francais, demander à un LLM de la traduire faire le RAG sur cette version traduite.
# On augmente et genere en anglais pour que par la suite un nouvelle agent re traduise la réponse en francais pour l'utilisateur final.

#1. Fonction de traduction du francais à l'anglais via LLM Groq

client = Groq(api_key= API_KEYS)

def fr_to_eng(req):
    
    message = [
        {"role": "user", "content": "the only task is to translate the following text from French to English. I juste want the translation: \n\n" + req}
    ]
    
    completion = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages= message,
    temperature=0.4,
    max_completion_tokens=1024,
    top_p=1,
    stream=True,
    stop=None
    )

    reponse = ""

    for chunk in completion:
        reponse += chunk.choices[0].delta.content or ""
    
    return reponse

print(fr_to_eng("Peux tu me donner des informations sur Douglas Siqueira Freitas ?"))
req_eng = fr_to_eng("Peux tu me donner des informations sur Douglas Siqueira Freitas ?")

#2. fonction d'augmentation_groq classique qui prend en entrée une requete en anglais et qui retourne les éléments à fournir au LLM pour répondre à la question
print(augmentation_groq(req_eng, 2))
aug_eng = augmentation_groq(req_eng, 2)

#3. fonction de génération_groq classique qui prend en entrée une requete en anglais ainsi que les éléments à fournir au LLM et qui retourne la réponse générée par le LLM
print(generation_gorq(req_eng, 2))
gen_eng = generation_gorq(req_eng, 2)

#4. fonction de traduction de l'anglais au francais via LLM Groq

def eng_to_fr(req):
    
    message = [
        {"role": "user", "content": "the only task is to translate the following text from English to French. I juste want the translation: \n\n" + req}
    ]
    
    completion = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages= message,
    temperature=0.4,
    max_completion_tokens=1024,
    top_p=1,
    stream=True,
    stop=None
    )

    reponse = ""

    for chunk in completion:
        reponse += chunk.choices[0].delta.content or ""
    
    return reponse

print(eng_to_fr(gen_eng))



Can you give me information about Douglas Siqueira Freitas?
[{'role': 'system', 'content': 'You are an assistant specializing in culinary recipes from around the world, created by university researchers.\nBased on these excerpts, please provide a detailed answer to the question and use only the information provided and give the page numbers.\n\nDo not try to give more information than what is provided.\n\nIf you cannot find an answer to the question, do not try to invent one.\n\n'}, {'role': 'user', 'content': 'This is the question :Can you give me information about Douglas Siqueira Freitas?\n\nHere are some excerpts to help you answer the question : \n\nExtract 1 \n\nPages : 18-19 \n\nTexte : INGREDIENTS PÃO DE  QUEIJO (Brazilian Cheese Bread) My favourite recipe – Brazil which can be bought in Germany 1 cup whole milk – Milch ½ cup vegetable oil – Pflanzenöl 1 tsp salt – Salz 2 cups tapioca or cassava flour – Tapioka-Stärke 2 large eggs – Eier 1 to 1½ cups grated Parmesan cheese  – P

Création de la fonction generation francais

In [62]:
#On va appliquer maintenant tout ce qui a été fait dans une fonction nommé generation_groq_fr

def generation_gorq_fr(req, top = 2):
    
    #1. Traduire la requete en anglais
    req_eng = fr_to_eng(req)
    
    #2. Retrieval et augmentation en anglais
    aug_eng = augmentation_groq(req_eng, top)
    
    #3. Generation en anglais
    gen_eng = generation_gorq(req_eng, top)
    
    #4. Traduction de la réponse en francais
    gen_fr = eng_to_fr(gen_eng)
    
    return gen_fr

print(generation_gorq_fr("Peux tu me donner des informations sur Douglas Siqueira Freitas ?", 2))

D'après les informations fournies dans le texte, Douglas Siqueira Freitas est un individu brésilien qui a été étudiant doctorant au Département de Science des Cultures, de Nutrition végétale et de Physiologie phylogénétique du 1er avril au 1er octobre 2017, avec une éventuelle extension d'autant de mois. Son projet de recherche portait sur "le Nickel dans les génotypes de soja". Le texte mentionne également qu'il a contribué une recette de Pain de Fromage brésilien (Pão de Queijo) à un livre de recettes international, indiquant qu'il a partagé sa recette préférée comme partie d'une collection de recettes de différents pays. (Page 18-19)


# ToolCalling de conversion d'unité de mesure

In [63]:
#creation de la liste de dico tools permettant de décrire chaque outils
tools = []

In [64]:
# On va créer une nouvelle fonction pour y applique du toolcalling qui explicité les conversions de mesures en d'autres unties en cuisine

def conversion_unite_mesure(valeur, unite_dep, unite_ar):
    
    if unite_dep == "gramme" and unite_ar == "onces":
        return valeur * 0.035274
    
    if unite_dep == "onces" and unite_ar == "gramme":
        return valeur * 28.3495
    
    if unite_dep == "litre" and unite_ar == "cups":
        return valeur * 4.22675
    
    if unite_dep == "cups" and unite_ar == "litre":
        return valeur * 0.236588
    
    if unite_dep == "celsius" and unite_ar == "fahrenheit":
        return (valeur * 9/5) + 32
    
    if unite_dep == "fahrenheit" and unite_ar == "celsius":
        return (valeur - 32) * 5/9
    

conversion_unite = {
    "type": "function",
    "function": {
        "name": "conversion_unite_mesure",
        "description": "Converts cooking measurement units (grams, ounces, liters, cups, celsius, fahrenheit)",
        "parameters": {
            "type": "object",
            "properties": {
                "valeur": {"type": "number", "description": "The quantity to convert"},
                "unite_dep": {"type": "string", "description": "The source unit (e.g: gram, liter, celsius)"},
                "unite_ar": {"type": "string", "description": "The target unit (e.g: ounces, cups, fahrenheit)"}
            }
        }
    }
}

tools.append(conversion_unite)

Connexion avec Groq

In [65]:
from groq import Groq
import json

client = Groq(api_key=API_KEYS)

def agent_groq(req):
    
    messages = [
        {"role": "system", "content": "You are a helpful cooking assistant. Use the tools available when needed."},
        {"role": "user", "content": req}
    ]
    
    # --- ÉTAPE 1 : On envoie la question + les outils au LLM ---
    completion = client.chat.completions.create(
        model="qwen/qwen3-32b",
        messages=messages,
        tools=tools,
        temperature=0.4,
        reasoning_effort="none",
        stream=False
    )
    
    response_message = completion.choices[0].message
    
    # --- ÉTAPE 2 : On vérifie si le LLM veut appeler un outil ---
    if response_message.tool_calls:
        
        # Le LLM a décidé d'appeler une fonction
        tool_call = response_message.tool_calls[0]
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        
        print(f"Le LLM veut appeler : {function_name}")
        print(f"Avec les arguments : {arguments}")
        
        # --- ÉTAPE 3 : On exécute la fonction Python nous-mêmes ---
        if function_name == "conversion_unite_mesure":
            resultat = conversion_unite_mesure(arguments["valeur"], arguments["unite_dep"], arguments["unite_ar"])
        
        print(f"Résultat de la fonction : {resultat}")
        
        # --- ÉTAPE 4 : On renvoie le résultat au LLM ---
        messages.append(response_message)  # on ajoute la réponse du LLM
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": str(resultat)
        })
        
        # --- ÉTAPE 5 : Le LLM formule la réponse finale ---
        final_completion = client.chat.completions.create(
            model="qwen/qwen3-32b",
            messages=messages,
            temperature=0.4,
            reasoning_effort="none",
            stream=False
        )
        
        return final_completion.choices[0].message.content
    
    else:
        # Le LLM n'a pas besoin d'outil, il répond directement
        return response_message.content

print(agent_groq("How many ounces is 500 grams?"))

Le LLM veut appeler : conversion_unite_mesure
Avec les arguments : {'unite_ar': 'ounces', 'unite_dep': 'gram', 'valeur': 500}
Résultat de la fonction : None
500 grams is approximately 17.64 ounces.


Connexion avec Groq et RAG

In [66]:
#creation du tool pour le RAG

rag_intern_recipe = {
    "type": "function",
    "function": {
        "name": "hybrid_search",
        "description": "Retrieves relevant excerpts from a world cuisine cookbook to answer questions about recipes, ingredients, techniques, and authors.",
        "parameters": {
            "type": "object",
            "properties": {
                "req": {"type": "string", "description": "The user's query in natural language"},
                "top": {"type": "integer", "description": "The number of relevant excerpts to retrieve"}
            }
        }
    }
}

tools.append(rag_intern_recipe)


In [67]:
#On va modifié ce code pour pouvoir aussi avoir notre RAG en tant qu'outil pour le LLM et pas seulement la fonction de conversion d'unité de mesure


from groq import Groq
import json

client = Groq(api_key=API_KEYS)

def agent_groq(req):
    
    messages = [
        {"role": "system", "content": "You are a helpful cooking assistant.Do not respond with bullet points, try to write in a natural way. Use the tools available when needed and provide detailed answers based on the information you can retrieve."},
        {"role": "user", "content": req}
    ]
    
    # --- ÉTAPE 1 : On envoie la question + les outils au LLM ---
    completion = client.chat.completions.create(
        model="qwen/qwen3-32b",
        messages=messages,
        tools=tools,
        temperature=0.4,
        reasoning_effort="none",
        stream=False
    )
    
    response_message = completion.choices[0].message
    
    # --- ÉTAPE 2 : On vérifie si le LLM veut appeler un outil ---
    if response_message.tool_calls:
        
        # Le LLM a décidé d'appeler une fonction
        tool_call = response_message.tool_calls[0]
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        
        print(f"Le LLM a appelé : {function_name}")
        print(f"A pris les arguments : {arguments}")
        
        # --- ÉTAPE 3 : On exécute la fonction Python nous-mêmes ---
        if function_name == "conversion_unite_mesure":
            resultat = conversion_unite_mesure(arguments["valeur"], arguments["unite_dep"], arguments["unite_ar"])
        
        
        elif function_name == "hybrid_search":
            resultat = hybrid_search(arguments["req"], arguments["top"])
        
        print(f"Résultat de la fonction : {resultat}")
        # --- ÉTAPE 4 : On renvoie le résultat au LLM ---
        messages.append(response_message)  # on ajoute la réponse du LLM
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": str(resultat)
        })
        
        # --- ÉTAPE 5 : Le LLM formule la réponse finale ---
        final_completion = client.chat.completions.create(
            model="qwen/qwen3-32b",
            messages=messages,
            temperature=0.4,
            reasoning_effort="none",
            stream=False
        )
        
        return final_completion.choices[0].message.content
    
    else:
        # Le LLM n'a pas besoin d'outil, il répond directement
        return f"Ugo a dit que je ne pouvais pas répondre à la question : {req}.\n\n Désolé !"

print(agent_groq("How many ounces is 500 grams?"))

Le LLM a appelé : conversion_unite_mesure
A pris les arguments : {'unite_ar': 'ounces', 'unite_dep': 'gram', 'valeur': 500}
Résultat de la fonction : None
500 grams is approximately 17.64 ounces.


TEST

In [68]:
print(agent_groq("What information do you have about Douglas Siqueira Freitas?"))

print("------------------------------------------")

print(agent_groq("What is the capital of France ?"))

Le LLM a appelé : hybrid_search
A pris les arguments : {'req': 'Douglas Siqueira Freitas', 'top': 5}
Résultat de la fonction : [{'page': '18-19', 'text': 'INGREDIENTS PÃO DE  QUEIJO (Brazilian Cheese Bread) My favourite recipe – Brazil which can be bought in Germany 1 cup whole milk – Milch ½ cup vegetable oil – Pflanzenöl 1 tsp salt – Salz 2 cups tapioca or cassava flour – Tapioka-Stärke 2 large eggs – Eier 1 to 1½ cups grated Parmesan cheese  – Parmesan Käse Equipment Medium saucepan Wooden spoon Standing mixer with paddle attachment  (or mixing bowl and elbow grease) 2 baking sheets Parchment paper or silicone baking mats Douglas Siqueira Freitas Brazilian Doctorate student  Department of crop science, plant nutrition  and phylogenetic physiology April – October 2017  (possible extension for more 6 months) Research project:  Nickel in soybean genotypes NOTES to the reciepe Tapioca flour: Sour cassava flour or sour  tapioca flour can be tricky to find. Look for it  at Latin American 